In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any
import math
import random
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

START_DEFAULT = (1, 2, 3,
                 5, 0, 6,
                 4, 7, 8)

GOAL_DEFAULT = (1, 2, 3,
                4, 5, 6,
                7, 8, 0)

ACTIONS = ["L", "R", "U", "D"]


@dataclass
class Node:
    id: str
    state: Tuple[int, ...]
    parent: Optional[str]
    action: Optional[str]
    depth: int
    h: int
    path: str


def parse_state(text: str) -> Tuple[int, ...]:
    text = text.replace("/", " ").replace(",", " ")
    nums = [int(x) for x in text.split()]

    if len(nums) != 9:
        raise ValueError("Trạng thái phải có đúng 9 số.")

    if sorted(nums) != list(range(9)):
        raise ValueError("Trạng thái phải chứa đủ các số từ 0 đến 8.")

    return tuple(nums)


def h_misplaced(state: Tuple[int, ...], goal: Tuple[int, ...]) -> int:
    return sum(1 for i, v in enumerate(state) if v != 0 and v != goal[i])


def state_line(state: Tuple[int, ...]) -> str:
    return " / ".join(
        " ".join(str(x) for x in state[i:i+3])
        for i in range(0, 9, 3)
    )


def move(state: Tuple[int, ...], action: str) -> Optional[Tuple[int, ...]]:
    zero = state.index(0)
    r, c = divmod(zero, 3)

    if action == "L":
        nr, nc = r, c - 1
    elif action == "R":
        nr, nc = r, c + 1
    elif action == "U":
        nr, nc = r - 1, c
    elif action == "D":
        nr, nc = r + 1, c
    else:
        return None

    if not (0 <= nr < 3 and 0 <= nc < 3):
        return None

    new_zero = nr * 3 + nc
    arr = list(state)
    arr[zero], arr[new_zero] = arr[new_zero], arr[zero]
    return tuple(arr)


def get_neighbors(state: Tuple[int, ...]) -> List[Tuple[str, Tuple[int, ...]]]:
    result = []
    for action in ACTIONS:
        nxt = move(state, action)
        if nxt is not None:
            result.append((action, nxt))
    return result


def node_number(node: Node) -> int:
    try:
        return int(node.id[1:])
    except Exception:
        return 0


def reconstruct_path(nodes: List[Node], goal_id: str) -> List[Node]:
    id_map = {n.id: n for n in nodes}
    path = []
    cur = id_map[goal_id]

    while True:
        path.append(cur)
        if cur.parent is None:
            break
        cur = id_map[cur.parent]

    return list(reversed(path))


# ============================================================
# 3. CSS + HTML GIAO DIỆN
# ============================================================

def css():
    display(HTML("""
    <style>
        :root {
            --bg: #f5f7fb;
            --panel: #ffffff;
            --panel-soft: #f8fbff;
            --text: #10213a;
            --muted: #5b6b83;
            --line: #dbe3ef;
            --primary: #153d8a;
            --primary-2: #0d1b4c;
            --accent: #7c5cff;
            --cyan: #15b8d6;
            --green: #1f9d62;
            --orange: #ea7d22;
            --red: #dc4d41;
        }

        * {
            font-family: "Segoe UI", Tahoma, Arial, sans-serif !important;
            box-sizing: border-box;
        }

        .app-wrap {
            background: var(--bg);
            padding: 8px 2px;
        }

        .app-title {
            background: linear-gradient(90deg, var(--primary-2), #0a2767 55%, var(--primary));
            color: white !important;
            padding: 24px 28px;
            border-radius: 26px;
            font-size: 30px;
            font-weight: 900;
            margin: 12px 0 16px 0;
            border: 2px solid rgba(255,255,255,.35);
            box-shadow: 0 10px 28px rgba(16, 33, 58, .16);
        }

        .app-subtitle {
            color: #d6e4ff !important;
            font-size: 14px;
            margin-top: 8px;
            font-weight: 500;
        }

        .section-title {
            background: linear-gradient(90deg, var(--primary-2), #09173f 55%, #0a245e);
            color: white !important;
            padding: 18px 22px;
            border-radius: 24px;
            font-size: 22px;
            font-weight: 850;
            margin: 14px 0 10px 0;
            border: 2px solid rgba(255,255,255,.30);
            box-shadow: 0 8px 22px rgba(16, 33, 58, .12);
        }

        .section-title span {
            border-left: 6px solid #a78bfa;
            padding-left: 14px;
            display: inline-block;
        }

        .panel {
            background: var(--panel);
            color: var(--text) !important;
            border: 1px solid var(--line);
            border-radius: 20px;
            padding: 16px;
            margin: 12px 0;
            box-shadow: 0 8px 24px rgba(16, 33, 58, .08);
        }

        .panel h2, .panel h3 {
            color: var(--text) !important;
            margin-top: 4px;
            margin-bottom: 10px;
        }

        .panel h2 {
            font-size: 22px;
        }

        .panel h3 {
            font-size: 18px;
        }

        .note, .success, .warn {
            border-radius: 14px;
            padding: 12px 14px;
            margin: 10px 0;
            font-size: 14px;
            border-left: 6px solid;
        }

        .note {
            background: #eef8ff;
            color: #0c5e84 !important;
            border-color: var(--cyan);
        }

        .success {
            background: #edf9f1;
            color: #196944 !important;
            border-color: var(--green);
        }

        .warn {
            background: #fff4ea;
            color: #9a4f11 !important;
            border-color: var(--orange);
        }

        .cards {
            display: flex;
            flex-wrap: wrap;
            gap: 14px;
            margin: 12px 0;
        }

        .card {
            background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%);
            color: var(--text) !important;
            border: 1px solid var(--line);
            border-top: 5px solid var(--primary);
            border-radius: 18px;
            padding: 12px;
            min-width: 174px;
            box-shadow: 0 6px 18px rgba(16, 33, 58, .08);
        }

        .card-title {
            text-align: center;
            font-weight: 800;
            color: var(--text) !important;
            margin-bottom: 8px;
        }

        .meta {
            text-align: center;
            color: var(--muted) !important;
            font-size: 12px;
            margin-top: 8px;
            line-height: 1.45;
        }

        table.matrix {
            border-collapse: separate;
            border-spacing: 5px;
            margin: auto;
        }

        table.matrix td {
            width: 38px;
            height: 38px;
            background: linear-gradient(135deg, #1b4fb5, #2cc2de);
            color: #ffffff !important;
            border-radius: 10px;
            font-size: 17px;
            font-weight: 800;
            text-align: center;
            vertical-align: middle;
            border: 1px solid rgba(16, 33, 58, .12);
        }

        table.matrix td.blank {
            background: #edf2f8;
            color: #97a4b5 !important;
            border: 1px dashed #bcc9d9;
        }

        .small-matrix td {
            width: 22px !important;
            height: 22px !important;
            font-size: 12px !important;
            border-radius: 7px !important;
        }

        table.pretty {
            width: 100%;
            border-collapse: collapse;
            background: white;
            color: var(--text) !important;
            margin: 12px 0;
            font-size: 13px;
            border-radius: 14px;
            overflow: hidden;
        }

        table.pretty th {
            background: #edf3ff;
            color: var(--text) !important;
            padding: 8px 6px;
            text-align: center;
            border: 1px solid var(--line);
            font-weight: 800;
            white-space: nowrap;
        }

        table.pretty td {
            padding: 7px 6px;
            border: 1px solid var(--line);
            color: var(--text) !important;
            text-align: center;
            vertical-align: top;
            background: white;
        }

        table.pretty tr:nth-child(even) td {
            background: #fafcff;
        }

        table.compact td, table.compact th {
            font-size: 12px;
            padding: 6px 5px;
        }

        .cell-state {
            text-align: left !important;
            white-space: nowrap;
            font-family: Consolas, monospace !important;
            font-size: 12px;
        }

        .cell-path {
            word-break: break-word;
            min-width: 90px;
        }

        .mini-node {
            background: #ffffff;
            border: 1px solid var(--line);
            border-radius: 12px;
            padding: 8px;
            min-width: 118px;
            display: inline-block;
            margin: 3px;
            vertical-align: top;
            color: var(--text) !important;
            box-shadow: 0 4px 12px rgba(16,33,58,.05);
            font-size: 12px;
        }

        .mini-wrap {
            display: flex;
            flex-wrap: wrap;
            gap: 6px;
            justify-content: center;
        }

        textarea, input {
            background: white !important;
            color: var(--text) !important;
            border: 1px solid #bcc9d9 !important;
            border-radius: 12px !important;
        }

        label, .widget-label {
            color: var(--text) !important;
            font-weight: 700 !important;
        }

        .widget-button button {
            border-radius: 14px !important;
            font-weight: 800 !important;
            letter-spacing: .1px;
            padding: 8px 16px !important;
            box-shadow: 0 5px 12px rgba(16,33,58,.08);
        }

        .widget-toggle-buttons button {
            border-radius: 12px !important;
            font-weight: 800 !important;
            min-width: 190px;
        }

        .compare-toolbar {
            margin: 8px 0 12px 0;
            padding: 10px 12px;
            border-radius: 14px;
            border: 1px solid var(--line);
            background: #f7faff;
        }

        .soft-caption {
            color: var(--muted) !important;
            font-size: 13px;
            margin-top: 3px;
        }

        .table-wrap {
            overflow-x: auto;
        }

        hr {
            border: none;
            border-top: 1px solid var(--line);
            margin: 18px 0;
        }
    </style>
    """))


def section_title_html(text: str) -> str:
    return f"<div class='section-title'><span>{text}</span></div>"


def matrix_html(state: Tuple[int, ...], small=False) -> str:
    cls = "matrix small-matrix" if small else "matrix"
    html = f"<table class='{cls}'>"
    for i in range(0, 9, 3):
        html += "<tr>"
        for v in state[i:i+3]:
            if v == 0:
                html += "<td class='blank'>0</td>"
            else:
                html += f"<td>{v}</td>"
        html += "</tr>"
    html += "</table>"
    return html


def state_card(state, title, h=None, node=None, action=None, border="#2563eb") -> str:
    meta = []
    if node is not None:
        meta.append(f"Node: <b>{node}</b>")
    if action is not None:
        meta.append(f"Action: <b>{action}</b>")
    if h is not None:
        meta.append(f"h = <b>{h}</b>")

    return f"""
    <div class="card" style="border-top-color:{border};">
        <div class="card-title">{title}</div>
        {matrix_html(state)}
        <div class="meta">{'<br>'.join(meta)}</div>
    </div>
    """


def cards_nodes(nodes: List[Node], color="#2563eb") -> str:
    if not nodes:
        return "<div class='note'>Không có node.</div>"

    html = "<div class='cards'>"
    for n in nodes:
        html += state_card(
            state=n.state,
            title=n.id,
            h=n.h,
            node=n.id,
            action=n.action if n.action else "Start",
            border=color
        )
    html += "</div>"
    return html


def cards_temp(items: List[Dict[str, Any]], color="#0ea5e9") -> str:
    if not items:
        return "<div class='note'>Không có trạng thái.</div>"

    html = "<div class='cards'>"
    for item in items:
        html += state_card(
            state=item["state"],
            title=item["title"],
            h=item["h"],
            node=item.get("id"),
            action=item.get("action"),
            border=color
        )
    html += "</div>"
    return html


def mini_node_html(n: Optional[Node]) -> str:
    if n is None:
        return "-"

    return f"""
    <div class='mini-node'>
        <b>{n.id}</b><br>
        Parent: {n.parent if n.parent else '-'}<br>
        Action: {n.action if n.action else 'Start'}<br>
        Depth: {n.depth}<br>
        h: {n.h}<br>
        Path: {n.path if n.path else '-'}<br>
        {matrix_html(n.state, small=True)}
    </div>
    """


def mini_temp_html(item: Dict[str, Any]) -> str:
    return f"""
    <div class='mini-node'>
        <b>{item.get('id', '-')}</b><br>
        Action: {item.get('action', '-')}<br>
        h: {item.get('h', '-')}<br>
        {matrix_html(item['state'], small=True)}
    </div>
    """


def node_list_html(nodes: List[Node]) -> str:
    if not nodes:
        return "-"
    return "<div class='mini-wrap'>" + "".join(mini_node_html(n) for n in nodes) + "</div>"


def temp_list_html(items: List[Dict[str, Any]]) -> str:
    if not items:
        return "-"
    return "<div class='mini-wrap'>" + "".join(mini_temp_html(item) for item in items) + "</div>"


def node_table_html(nodes: List[Node]) -> str:
    html = """
    <div class='table-wrap'>
    <table class="pretty compact">
        <tr>
            <th>Node</th>
            <th>Parent</th>
            <th>Action</th>
            <th>Depth</th>
            <th>h</th>
            <th>Path</th>
            <th>State</th>
        </tr>
    """

    for n in nodes:
        html += f"""
        <tr>
            <td><b>{n.id}</b></td>
            <td>{n.parent if n.parent else '-'}</td>
            <td>{n.action if n.action else 'Start'}</td>
            <td>{n.depth}</td>
            <td><b>{n.h}</b></td>
            <td class='cell-path'>{n.path if n.path else '-'}</td>
            <td class='cell-state'>{state_line(n.state)}</td>
        </tr>
        """

    html += "</table></div>"
    return html


def reached_short_html(reached: List[Tuple[int, ...]], goal: Tuple[int, ...], max_show=8) -> str:
    if not reached:
        return "-"

    html = f"<b>{len(reached)} trạng thái</b><br>"
    html += "<div class='mini-wrap'>"

    for i, s in enumerate(reached[:max_show]):
        html += f"""
        <div class='mini-node'>
            <b>R{i}</b><br>
            h = {h_misplaced(s, goal)}
            {matrix_html(s, small=True)}
        </div>
        """

    html += "</div>"
    if len(reached) > max_show:
        html += f"<div class='soft-caption'>... còn {len(reached) - max_show} trạng thái nữa</div>"
    return html


# ============================================================
# 4. THUẬT TOÁN
# ============================================================

def beam_search(start: Tuple[int, ...], goal: Tuple[int, ...], k=2, max_levels=30):
    node_id = 0
    start_node = Node(
        id=f"N{node_id}", state=start, parent=None, action=None,
        depth=0, h=h_misplaced(start, goal), path=""
    )
    node_id += 1

    nodes = [start_node]
    frontier = [start_node]
    reached = {start}
    reached_order = [start]
    steps = []
    goal_node = None

    for level in range(max_levels + 1):
        frontier_before = list(frontier)
        expanded = []
        generated = []

        for n in frontier_before:
            if n.state == goal:
                goal_node = n
                break

        if goal_node:
            steps.append({
                "algo": "beam",
                "step": level,
                "frontier_before": frontier_before,
                "expanded": [],
                "generated": [],
                "selected": frontier_before,
                "reached": list(reached_order),
                "note": f"Goal đã nằm trong Frontier tại {goal_node.id}."
            })
            return goal_node, nodes, steps

        candidates = []

        for parent in frontier_before:
            expanded.append(parent)
            for action, nxt in get_neighbors(parent.state):
                if nxt in reached:
                    continue

                child = Node(
                    id=f"N{node_id}", state=nxt, parent=parent.id, action=action,
                    depth=parent.depth + 1, h=h_misplaced(nxt, goal), path=parent.path + action
                )
                node_id += 1

                nodes.append(child)
                generated.append(child)
                candidates.append(child)
                reached.add(nxt)
                reached_order.append(nxt)

                if nxt == goal:
                    goal_node = child

        candidates.sort(key=lambda n: (n.h, n.depth, node_number(n)))
        selected = candidates[:k]

        if goal_node:
            selected = [goal_node]
            note = f"Sinh ra Goal tại {goal_node.id}. Dừng thuật toán."
        elif selected:
            note = "Chọn k node có h nhỏ nhất: " + ", ".join(f"{n.id}(h={n.h})" for n in selected)
        else:
            note = "Không sinh được node mới."

        steps.append({
            "algo": "beam",
            "step": level,
            "frontier_before": frontier_before,
            "expanded": expanded,
            "generated": generated,
            "selected": selected,
            "reached": list(reached_order),
            "note": note
        })

        if goal_node or not selected:
            break
        frontier = selected

    return goal_node, nodes, steps


def simulated_annealing(start: Tuple[int, ...], goal: Tuple[int, ...], T0=5.0, alpha=0.8,
                        Tmin=0.01, max_steps=100, seed=0):
    random.seed(seed)
    node_id = 0

    current = Node(
        id=f"N{node_id}", state=start, parent=None, action=None,
        depth=0, h=h_misplaced(start, goal), path=""
    )
    node_id += 1

    nodes = [current]
    reached = {start}
    reached_order = [start]
    steps = []
    T = T0

    for step in range(max_steps):
        if current.state == goal or T <= Tmin:
            break

        current_before = current
        frontier = []
        for action, nxt in get_neighbors(current.state):
            frontier.append({
                "id": f"F{step}_{action}",
                "title": f"Đi {action}",
                "action": action,
                "state": nxt,
                "h": h_misplaced(nxt, goal)
            })

        chosen = random.choice(frontier)
        delta = chosen["h"] - current_before.h

        if delta < 0:
            p = 1.0
            r = None
            accepted = True
            note = "Δ < 0 nên tốt hơn, nhận luôn."
        else:
            p = math.exp(-delta / T)
            r = random.random()
            accepted = r < p
            note = "Nhận theo xác suất p." if accepted else "Không nhận vì r ≥ p."

        new_node = None
        if accepted:
            new_node = Node(
                id=f"N{node_id}", state=chosen["state"], parent=current_before.id,
                action=chosen["action"], depth=current_before.depth + 1,
                h=chosen["h"], path=current_before.path + chosen["action"]
            )
            node_id += 1
            nodes.append(new_node)
            current = new_node

        if chosen["state"] not in reached:
            reached.add(chosen["state"])
            reached_order.append(chosen["state"])

        steps.append({
            "algo": "sa",
            "step": step,
            "T": T,
            "current_before": current_before,
            "frontier": frontier,
            "chosen": chosen,
            "delta": delta,
            "p": p,
            "r": r,
            "accepted": accepted,
            "new_node": new_node,
            "current_after": current,
            "reached": list(reached_order),
            "note": note
        })

        T *= alpha
        if current.state == goal:
            break

    goal_node = current if current.state == goal else None
    return goal_node, nodes, steps


# ============================================================
# 5. BẢNG CHI TIẾT VÀ RENDER
# ============================================================

def beam_trace_table_html(steps, goal):
    html = """
    <div class='panel'>
    <h2>Bảng chạy Beam Search chi tiết</h2>
    <div class='table-wrap'>
    <table class='pretty compact'>
        <tr>
            <th>Tầng</th>
            <th>Node đang xét</th>
            <th>Frontier trước</th>
            <th>Node con sinh ra</th>
            <th>Frontier sau chọn k</th>
            <th>Reached</th>
            <th>Ghi chú</th>
        </tr>
    """

    for st in steps:
        html += f"""
        <tr>
            <td><b>{st['step']}</b></td>
            <td>{node_list_html(st['expanded'])}</td>
            <td>{node_list_html(st['frontier_before'])}</td>
            <td>{node_list_html(st['generated'])}</td>
            <td>{node_list_html(st['selected'])}</td>
            <td>{reached_short_html(st['reached'], goal)}</td>
            <td style='text-align:left;'>{st['note']}</td>
        </tr>
        """

    html += "</table></div></div>"
    return html


def sa_trace_table_html(steps, goal):
    html = """
    <div class='panel'>
    <h2>Bảng chạy Simulated Annealing chi tiết</h2>
    <div class='table-wrap'>
    <table class='pretty compact'>
        <tr>
            <th>Bước</th>
            <th>T</th>
            <th>Node hiện tại</th>
            <th>Frontier</th>
            <th>Next được chọn</th>
            <th>Δ</th>
            <th>p</th>
            <th>r</th>
            <th>Kết quả</th>
            <th>Node sau bước</th>
            <th>Reached</th>
        </tr>
    """

    for st in steps:
        r_text = "-" if st["r"] is None else f"{st['r']:.4f}"
        accepted_text = "Nhận" if st["accepted"] else "Không nhận"
        html += f"""
        <tr>
            <td><b>{st['step']}</b></td>
            <td>{st['T']:.4f}</td>
            <td>{mini_node_html(st['current_before'])}</td>
            <td>{temp_list_html(st['frontier'])}</td>
            <td>{temp_list_html([st['chosen']])}</td>
            <td><b>{st['delta']}</b></td>
            <td>{st['p']:.4f}</td>
            <td>{r_text}</td>
            <td><b>{accepted_text}</b><br>{st['note']}</td>
            <td>{mini_node_html(st['current_after'])}</td>
            <td>{reached_short_html(st['reached'], goal)}</td>
        </tr>
        """

    html += "</table></div></div>"
    return html


def render_beam_step(record, goal):
    html = f"""
    <div class='panel'>
        <h2>Beam Search - Tầng {record['step']}</h2>
        <div class='note'>{record['note']}</div>
        <h3>1. Node đang xét</h3>
        {cards_nodes(record['expanded'], color='#1d4ed8')}
        <h3>2. Frontier trước khi mở rộng</h3>
        {cards_nodes(record['frontier_before'], color='#15b8d6')}
        <h3>3. Các node con sinh ra</h3>
        {cards_nodes(record['generated'], color='#ea7d22')}
        <h3>4. Frontier mới sau khi chọn k node tốt nhất</h3>
        {cards_nodes(record['selected'], color='#1f9d62')}
        <h3>5. Reached</h3>
        {reached_short_html(record['reached'], goal, max_show=18)}
    </div>
    """
    display(HTML(html))


def render_sa_step(record, goal):
    chosen = record['chosen']
    r_text = "-" if record['r'] is None else f"{record['r']:.4f}"
    result = "NHẬN" if record['accepted'] else "KHÔNG NHẬN"

    html = f"""
    <div class='panel'>
        <h2>Simulated Annealing - Bước {record['step']}</h2>
        <div class='note'>
            T = <b>{record['T']:.4f}</b><br>
            Δ = h(next) - h(current) = <b>{record['delta']}</b><br>
            p = <b>{record['p']:.4f}</b>, r = <b>{r_text}</b><br>
            Kết quả: <b>{result}</b><br>
            {record['note']}
        </div>
        <h3>1. Node hiện tại</h3>
        {cards_nodes([record['current_before']], color='#1d4ed8')}
        <h3>2. Frontier: các hàng xóm có thể chọn ngẫu nhiên</h3>
        {cards_temp(record['frontier'], color='#15b8d6')}
        <h3>3. Next được chọn ngẫu nhiên</h3>
        {cards_temp([{ 'id': chosen['id'], 'title': 'Next được chọn', 'action': chosen['action'], 'state': chosen['state'], 'h': chosen['h']}], color='#ea7d22')}
        <h3>4. Node hiện tại sau bước này</h3>
        {cards_nodes([record['current_after']], color='#1f9d62' if record['accepted'] else '#64748b')}
        <h3>5. Reached</h3>
        {reached_short_html(record['reached'], goal, max_show=18)}
    </div>
    """
    display(HTML(html))


def solution_html(path_nodes, title):
    html = f"<div class='panel'><h2>{title}</h2><div class='cards'>"
    for i, n in enumerate(path_nodes):
        html += state_card(
            state=n.state,
            title='Start' if i == 0 else f'Bước {i}: {n.action}',
            h=n.h,
            node=n.id,
            action=n.action if n.action else 'Start',
            border='#1f9d62' if n.h == 0 else '#1d4ed8'
        )
    html += "</div></div>"
    return html


# ============================================================
# 6. APP WIDGET
# ============================================================

css()

display(HTML("""
<div class='app-wrap'>
    <div class='app-title'>
        8-Puzzle Solver | Lê Minh Huy
        <div class='app-subtitle'>Giao diện nền sáng, có nút chuyển đổi giữa các thuật toán và bảng Node hiển thị gọn hơn nhưng vẫn đầy đủ thông tin.</div>
    </div>
</div>
"""))

shared_text_layout = widgets.Layout(width='360px', height='110px')
shared_slider_layout = widgets.Layout(width='47%')
shared_small_layout = widgets.Layout(width='180px')

start_input = widgets.Textarea(
    value="1 2 3\n5 0 6\n4 7 8",
    description='Start',
    layout=shared_text_layout,
    style={'description_width': '60px'}
)

goal_input = widgets.Textarea(
    value="1 2 3\n4 5 6\n7 8 0",
    description='Goal',
    layout=shared_text_layout,
    style={'description_width': '60px'}
)

k_slider = widgets.IntSlider(
    value=2, min=1, max=5, step=1, description='Beam k',
    layout=shared_slider_layout, style={'description_width': '90px'}
)

max_level_slider = widgets.IntSlider(
    value=30, min=1, max=100, step=1, description='Max level',
    layout=shared_slider_layout, style={'description_width': '90px'}
)

T0_input = widgets.FloatText(
    value=5.0, description='T0', layout=shared_small_layout,
    style={'description_width': '60px'}
)

alpha_slider = widgets.FloatSlider(
    value=0.8, min=0.1, max=0.99, step=0.01, description='alpha',
    layout=widgets.Layout(width='260px'), style={'description_width': '70px'}
)

Tmin_input = widgets.FloatText(
    value=0.01, description='Tmin', layout=shared_small_layout,
    style={'description_width': '60px'}
)

seed_input = widgets.IntText(
    value=0, description='Seed', layout=shared_small_layout,
    style={'description_width': '60px'}
)

max_steps_slider = widgets.IntSlider(
    value=100, min=1, max=300, step=1, description='Max steps',
    layout=widgets.Layout(width='47%'), style={'description_width': '100px'}
)

run_beam_btn = widgets.Button(description='Chạy Beam Search', button_style='info', icon='play')
run_sa_btn = widgets.Button(description='Chạy Simulated Annealing', button_style='success', icon='play')
run_both_btn = widgets.Button(description='So sánh cả hai', button_style='warning', icon='exchange')

algo_switch = widgets.ToggleButtons(
    options=[('Beam Search', 'beam'), ('Simulated Annealing', 'sa')],
    description='Hiển thị:',
    value='beam',
    layout=widgets.Layout(width='auto'),
    style={'description_width': '80px'}
)
algo_switch.layout.display = 'none'

first_btn = widgets.Button(description='⏮ Đầu')
prev_btn = widgets.Button(description='← Trước')
next_btn = widgets.Button(description='Sau →')
last_btn = widgets.Button(description='Cuối ⏭')

status = widgets.HTML("<b style='color:#10213a;'>Trạng thái:</b> Chưa chạy thuật toán.")
summary_out = widgets.Output()
step_out = widgets.Output()

APP = {
    'view_mode': 'single',
    'active_algo': 'beam',
    'start': START_DEFAULT,
    'goal': GOAL_DEFAULT,
    'comparison_ready': False,
    'beam': {'label': 'Beam Search', 'goal_node': None, 'nodes': [], 'steps': [], 'index': 0},
    'sa': {'label': 'Simulated Annealing', 'goal_node': None, 'nodes': [], 'steps': [], 'index': 0},
}


def get_input_states():
    start = parse_state(start_input.value)
    goal = parse_state(goal_input.value)
    return start, goal


def preview_html(start, goal):
    return HTML(
        "<div class='panel'><h2>Start và Goal</h2><div class='cards'>"
        + state_card(start, 'Start', h=h_misplaced(start, goal), border='#1d4ed8')
        + state_card(goal, 'Goal', h=0, border='#1f9d62')
        + "</div></div>"
    )


def compare_table_html():
    beam = APP['beam']
    sa = APP['sa']
    beam_path = ' → '.join(beam['goal_node'].path) if beam['goal_node'] else '-'
    sa_path = ' → '.join(sa['goal_node'].path) if sa['goal_node'] else '-'
    return f"""
    <div class='panel'>
        <h2>So sánh 2 thuật toán</h2>
        <div class='soft-caption'>Dùng nút chuyển đổi bên dưới để xem chi tiết từng thuật toán.</div>
        <div class='table-wrap'>
        <table class='pretty compact'>
            <tr>
                <th>Thuật toán</th>
                <th>Tìm thấy Goal</th>
                <th>Đường đi</th>
                <th>Số Node</th>
                <th>Số bước trace</th>
            </tr>
            <tr>
                <td><b>Beam Search</b></td>
                <td>{'Có' if beam['goal_node'] else 'Không'}</td>
                <td>{beam_path}</td>
                <td>{len(beam['nodes'])}</td>
                <td>{len(beam['steps'])}</td>
            </tr>
            <tr>
                <td><b>Simulated Annealing</b></td>
                <td>{'Có' if sa['goal_node'] else 'Không'}</td>
                <td>{sa_path}</td>
                <td>{len(sa['nodes'])}</td>
                <td>{len(sa['steps'])}</td>
            </tr>
        </table>
        </div>
    </div>
    """


def active_pack():
    return APP[APP['active_algo']]


def show_summary_for_active():
    pack = active_pack()
    mode = pack['label']
    goal_node = pack['goal_node']
    nodes = pack['nodes']
    steps = pack['steps']
    start = APP['start']
    goal = APP['goal']

    with summary_out:
        clear_output()
        css()
        display(preview_html(start, goal))

        if APP['comparison_ready']:
            display(HTML(compare_table_html()))
            display(HTML("<div class='compare-toolbar'><b>Đang xem chi tiết:</b> " + mode + "</div>"))

        display(HTML(f"<div class='panel'><h2>Kết quả {mode}</h2></div>"))

        if goal_node:
            path_nodes = reconstruct_path(nodes, goal_node.id)
            path_text = ' → '.join(goal_node.path) if goal_node.path else '(đã là Goal)'
            display(HTML(f"""
            <div class='success'>
                <b>Tìm thấy Goal!</b><br>
                Node Goal: <b>{goal_node.id}</b><br>
                Đường đi: <b>{path_text}</b><br>
                Số bước: <b>{len(path_nodes) - 1}</b>
            </div>
            """))
            display(HTML(solution_html(path_nodes, f"Đường đi lời giải - {mode}")))
        else:
            display(HTML("""
            <div class='warn'>
                Chưa tìm thấy Goal trong giới hạn đã đặt.
                Hãy tăng Max level / Max steps hoặc đổi Seed.
            </div>
            """))

        display(HTML("<div class='panel'><h2>Bảng Node (nhỏ gọn, đầy đủ)</h2>"))
        display(HTML(node_table_html(nodes)))
        display(HTML("</div>"))

        if mode == 'Beam Search':
            display(HTML(beam_trace_table_html(steps, goal)))
        else:
            display(HTML(sa_trace_table_html(steps, goal)))

        display(HTML("""
        <div class='note'>
            Dùng các nút <b>Đầu</b>, <b>Trước</b>, <b>Sau</b>, <b>Cuối</b>
            để xem mô phỏng từng bước của thuật toán đang được chọn.
        </div>
        """))


def draw_step():
    pack = active_pack()
    with step_out:
        clear_output()
        css()
        if not pack['steps']:
            display(HTML("<div class='warn'>Chưa có bước chạy nào.</div>"))
            return

        idx = pack['index']
        total = len(pack['steps'])
        record = pack['steps'][idx]
        status.value = (
            f"<b style='color:#10213a;'>Đang xem:</b> {pack['label']} - bước {idx + 1}/{total}"
        )

        if record['algo'] == 'beam':
            render_beam_step(record, APP['goal'])
        else:
            render_sa_step(record, APP['goal'])


def refresh_active_views():
    show_summary_for_active()
    draw_step()


def load_single_result(algo_key, label, goal_node, nodes, steps, start, goal):
    APP['view_mode'] = 'single'
    APP['active_algo'] = algo_key
    APP['start'] = start
    APP['goal'] = goal
    APP['comparison_ready'] = False
    algo_switch.layout.display = 'none'

    APP[algo_key] = {
        'label': label,
        'goal_node': goal_node,
        'nodes': nodes,
        'steps': steps,
        'index': 0,
    }
    refresh_active_views()


def load_compare_results(start, goal, beam_goal, beam_nodes, beam_steps, sa_goal, sa_nodes, sa_steps):
    APP['view_mode'] = 'compare'
    APP['active_algo'] = algo_switch.value
    APP['start'] = start
    APP['goal'] = goal
    APP['comparison_ready'] = True
    algo_switch.layout.display = 'flex'

    APP['beam'] = {
        'label': 'Beam Search',
        'goal_node': beam_goal,
        'nodes': beam_nodes,
        'steps': beam_steps,
        'index': 0,
    }
    APP['sa'] = {
        'label': 'Simulated Annealing',
        'goal_node': sa_goal,
        'nodes': sa_nodes,
        'steps': sa_steps,
        'index': 0,
    }
    refresh_active_views()


def run_beam(_):
    try:
        start, goal = get_input_states()
        goal_node, nodes, steps = beam_search(start=start, goal=goal, k=k_slider.value, max_levels=max_level_slider.value)
        load_single_result('beam', 'Beam Search', goal_node, nodes, steps, start, goal)
    except Exception as e:
        with summary_out:
            clear_output()
            display(HTML(f"<div class='warn'><b>Lỗi:</b> {e}</div>"))


def run_sa(_):
    try:
        start, goal = get_input_states()
        goal_node, nodes, steps = simulated_annealing(
            start=start, goal=goal, T0=T0_input.value, alpha=alpha_slider.value,
            Tmin=Tmin_input.value, max_steps=max_steps_slider.value, seed=seed_input.value
        )
        load_single_result('sa', 'Simulated Annealing', goal_node, nodes, steps, start, goal)
    except Exception as e:
        with summary_out:
            clear_output()
            display(HTML(f"<div class='warn'><b>Lỗi:</b> {e}</div>"))


def run_both(_):
    try:
        start, goal = get_input_states()
        beam_goal, beam_nodes, beam_steps = beam_search(start=start, goal=goal, k=k_slider.value, max_levels=max_level_slider.value)
        sa_goal, sa_nodes, sa_steps = simulated_annealing(
            start=start, goal=goal, T0=T0_input.value, alpha=alpha_slider.value,
            Tmin=Tmin_input.value, max_steps=max_steps_slider.value, seed=seed_input.value
        )
        load_compare_results(start, goal, beam_goal, beam_nodes, beam_steps, sa_goal, sa_nodes, sa_steps)
    except Exception as e:
        with summary_out:
            clear_output()
            display(HTML(f"<div class='warn'><b>Lỗi:</b> {e}</div>"))


def prev_step(_):
    pack = active_pack()
    if pack['steps']:
        pack['index'] = max(0, pack['index'] - 1)
        draw_step()


def next_step(_):
    pack = active_pack()
    if pack['steps']:
        pack['index'] = min(len(pack['steps']) - 1, pack['index'] + 1)
        draw_step()


def first_step(_):
    pack = active_pack()
    if pack['steps']:
        pack['index'] = 0
        draw_step()


def last_step(_):
    pack = active_pack()
    if pack['steps']:
        pack['index'] = len(pack['steps']) - 1
        draw_step()


def on_algo_switch(change):
    if change['name'] == 'value' and change['new']:
        APP['active_algo'] = change['new']
        refresh_active_views()


run_beam_btn.on_click(run_beam)
run_sa_btn.on_click(run_sa)
run_both_btn.on_click(run_both)
first_btn.on_click(first_step)
prev_btn.on_click(prev_step)
next_btn.on_click(next_step)
last_btn.on_click(last_step)
algo_switch.observe(on_algo_switch, names='value')


row_inputs = widgets.HBox([start_input, goal_input], layout=widgets.Layout(flex_flow='row wrap', gap='16px'))
row_beam = widgets.HBox([k_slider, max_level_slider], layout=widgets.Layout(flex_flow='row wrap', gap='20px'))
row_sa1 = widgets.HBox([T0_input, alpha_slider, Tmin_input, seed_input], layout=widgets.Layout(flex_flow='row wrap', gap='16px', align_items='center'))
row_buttons = widgets.HBox([run_beam_btn, run_sa_btn, run_both_btn], layout=widgets.Layout(flex_flow='row wrap', gap='12px'))
row_step = widgets.HBox([first_btn, prev_btn, next_btn, last_btn], layout=widgets.Layout(flex_flow='row wrap', gap='10px'))

ui = widgets.VBox([
    widgets.HTML(section_title_html('1. Nhập trạng thái bài toán')),
    row_inputs,

    widgets.HTML(section_title_html('2. Tham số Beam Search')),
    row_beam,

    widgets.HTML(section_title_html('3. Tham số Simulated Annealing')),
    row_sa1,
    max_steps_slider,

    widgets.HTML(section_title_html('4. Chạy và so sánh')),
    row_buttons,
    algo_switch,

    widgets.HTML(section_title_html('5. Điều khiển mô phỏng từng bước')),
    row_step,
    status,

    widgets.HTML("<hr><h2 style='color:#10213a;'>Kết quả tổng quát</h2>"),
    summary_out,

    widgets.HTML("<hr><h2 style='color:#10213a;'>Mô phỏng từng bước: Node - Frontier - Reached</h2>"),
    step_out
])

display(ui)
